In [ ]:
!pip install yfinance -q
import yfinance as yf
import pandas as pd
import os
import warnings
import time
from datetime import date

# Suppress warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# ==========================================
# 1. SETTINGS
# ==========================================
LOOKBACK_DAYS = 80
TOP_PCT       = 25
BTM_PCT       = 25
FORCE_UPDATE  = False 

# SPECIAL MAPPING (TradingView Symbol -> Yahoo Symbol)
TICKER_MAP = {
    "SPX": "^SPX",
    "XSP": "^XSP",
    "DJI": "^DJI",
    "IXIC": "^IXIC",
    "GPS": "GAP",
    "BRK.B": "BRK-B"
}

# PASTE YOUR LIST HERE
RAW_LIST = """
A
AA
AAL
AAP
AAPL
ABBV
ABNB
ABT
ACN
ADBE
ADI
ADM
ADP
ADSK
AEP
AES
AFL
AFRM
AGNC
AI
AIG
AKAM
ALB
ALK
ALL
AMAT
AMD
AMGN
AMRN
AMZN
APA
APH
APTV
AVGO
AXP
BA
BABA
BAC
BAX
BBY
BEN
BIDU
BIIB
BITO
BK
BKR
BMY
BP
BSX
BX
BYND
C
CAH
CAT
CB
CCI
CCJ
CCL
CF
CFG
CHWY
CI
CL
CLF
CLX
CMCSA
CME
CNC
CNP
COF
COIN
COP
COST
CPB
CPRI
CRM
CRON
CRWD
CSCO
CSX
CTVA
CVNA
CVS
CVX
CZR
D
DAL
DD
DE
DELL
DHI
DHR
DIA
DIS
DKNG
DLR
DOCU
DOW
DRI
DVN
DXC
EA
EBAY
ED
EEM
EFA
EIX
EL
EMR
ENPH
EOG
EQR
EQT
ETSY
EVRG
EW
EWJ
EWW
EWY
EWZ
EXC
EXPE
F
FANG
FAST
FCX
FDX
FE
FI
FITB
FOXA
FSLR
FTI
FTV
FXE
FXI
GD
GDX
GE
GILD
GLD
GLW
GM
GME
GOOG
GOOGL
GPRO
GPS
GS
HAL
HBAN
HBI
HCA
HD
HIG
HLT
HOG
HOLX
HON
HPE
HPQ
HYG
IBB
IBIT
IBM
ICE
IEF
INTC
IP
IPG
IRM
IVZ
IWM
IYR
JCI
JD
JETS
JNJ
JNPR
JPM
K
KHC
KMI
KO
KR
KRE
KSS
KWEB
LEN
LI
LKQ
LLY
LNC
LOW
LQD
LUMN
LUV
LVS
LYB
LYFT
MA
MAR
MARA
MAS
MCD
MCHP
MDLZ
MDT
MET
META
MGM
MMM
MO
MOS
MPC
MRK
MRNA
MRVL
MS
MSFT
MSTR
MTB
MTCH
MU
NCLH
NEE
NEM
NET
NFLX
NKE
NOW
NRG
NTAP
NTES
NVAX
NVDA
NWL
NWSA
O
OIH
OKE
OMC
ON
ORCL
OXY
PARA
PBR
PDD
PENN
PEP
PFE
PFG
PG
PGR
PINS
PLTR
PNC
PPL
PRU
PSX
PTON
PYPL
QCOM
QQQ
RBLX
RCL
RF
RIG
RIOT
RIVN
ROKU
ROST
RTX
RUN
SBUX
SCHD
SCHW
SEDG
SHOP
SIRI
SLB
SLV
SMCI
SMH
SNAP
SNOW
SO
SOFI
SOXL
SOXS
SPG
SPX
SPXL
SPXS
SPY
SQQQ
STX
SWKS
SYF
SYY
T
TAP
TBT
TCOM
TDOC
TFC
TGT
TJX
TLT
TMO
TMUS
TPR
TQQQ
TRIP
TRV
TSLA
TSM
TSN
TT
TTD
TTWO
TXN
U
UA
UAA
UAL
UBER
ULTA
UNG
UNH
UNM
UNP
UPS
UPST
URBN
USB
USO
UVXY
V
VALE
VFC
VGK
VTR
VXX
VZ
WAB
WBA
WBD
WDC
WFC
WM
WMB
WMT
WU
WY
WYNN
X
XBI
XEL
XHB
XLB
XLC
XLE
XLF
XLI
XLK
XLP
XLRE
XLU
XLV
XLY
XOM
XOP
XRT
XRX
XSP
XYZ
YELP
YINN
ZION
ZM
"""

# ==========================================
# 2. DATA PROCESSING
# ==========================================
def get_clean_tickers():
    raw_tickers = [t.strip() for t in RAW_LIST.split('\n') if t.strip()]
    raw_tickers = list(set(raw_tickers))
    
    yahoo_tickers = []
    reverse_map = {} 
    
    for t in raw_tickers:
        y_tick = TICKER_MAP.get(t, t) 
        if t not in TICKER_MAP:
            y_tick = y_tick.replace('.', '-')
        yahoo_tickers.append(y_tick)
        reverse_map[y_tick] = t
        
    return yahoo_tickers, reverse_map

def get_market_data(tickers):
    today_str = date.today().strftime("%Y-%m-%d")
    filename = f'my_watchlist_data_{today_str}.pkl'
    
    if os.path.exists(filename) and not FORCE_UPDATE:
        print(f"   [CACHE] Loading Data from {filename}...")
        return pd.read_pickle(filename)
    
    print(f"   [WEB] Phase 1: Bulk Downloading {len(tickers)} symbols...")
    
    # 1. Bulk Download
    data = yf.download(tickers, period="6mo", interval="1d", progress=True, auto_adjust=True, threads=True)['Close']
    
    # 2. Check for Failures
    downloaded = list(data.columns)
    missing = list(set(tickers) - set(downloaded))
    
    # 3. Smart Retry Logic
    if missing:
        print(f"\n   [RETRY] Phase 2: Retrying {len(missing)} failed symbols individually...")
        
        for failed_ticker in missing:
            try:
                # Small delay to be polite to API
                time.sleep(0.2)
                single_data = yf.download(failed_ticker, period="6mo", interval="1d", progress=False, auto_adjust=True)
                
                if not single_data.empty and 'Close' in single_data.columns:
                    # Merge into main dataframe
                    # We treat the single series as a new column
                    data[failed_ticker] = single_data['Close']
                    print(f"      -> Recovered: {failed_ticker}")
                else:
                    print(f"      -> Permanently Failed: {failed_ticker}")
            except Exception as e:
                print(f"      -> Error on {failed_ticker}: {e}")

    # Save to cache
    data.to_pickle(filename)
    return data

# ==========================================
# 3. MAIN SCANNER
# ==========================================
def run_scanner():
    print("--- STEP 1: PROCESSING TICKER LIST ---")
    yahoo_tickers, reverse_map = get_clean_tickers()
    print(f"   Loaded {len(yahoo_tickers)} unique symbols.")
    
    print("\n--- STEP 2: MARKET DATA ---")
    data = get_market_data(yahoo_tickers)
    
    print("\n--- STEP 3: ANALYZING ZONES ---")
    recent_data = data.tail(LOOKBACK_DAYS)
    
    premium_list = []
    discount_list = []
    
    for y_ticker in recent_data.columns:
        try:
            series = recent_data[y_ticker].dropna()
            if series.empty: continue
            
            curr = series.iloc[-1]
            hi   = series.max()
            lo   = series.min()
            rng  = hi - lo
            
            if rng == 0: continue
            
            loc = (curr - lo) / rng
            
            tv_ticker = reverse_map.get(y_ticker, y_ticker)
            
            item = {
                'ticker': tv_ticker, 
                'loc': round(loc*100, 1), 
                'price': round(curr, 2)
            }
            
            if loc >= (1.0 - (TOP_PCT/100)):
                premium_list.append(item)
            elif loc <= (BTM_PCT/100):
                discount_list.append(item)
                
        except Exception:
            continue

    # ==========================================
    # OUTPUT
    # ==========================================
    print("\n" + "="*50)
    print(f"DISCOUNT ZONES (Bottom {BTM_PCT}%) - {len(discount_list)} Stocks")
    print("="*50)
    discount_list.sort(key=lambda x: x['loc'])
    for s in discount_list[:20]: 
        print(f"{s['ticker']:<8} {s['loc']:<6}% ${s['price']}")
    if len(discount_list) > 20: print(f"... and {len(discount_list)-20} more.")

    print("\n" + "="*50)
    print(f"PREMIUM ZONES (Top {TOP_PCT}%) - {len(premium_list)} Stocks")
    print("="*50)
    premium_list.sort(key=lambda x: x['loc'], reverse=True)
    for s in premium_list[:20]:
        print(f"{s['ticker']:<8} {s['loc']:<6}% ${s['price']}")
    if len(premium_list) > 20: print(f"... and {len(premium_list)-20} more.")

    # ==========================================
    # TRADINGVIEW IMPORT STRINGS
    # ==========================================
    print("\n\n" + "#"*60)
    print("TRADINGVIEW IMPORT STRINGS (COPY & PASTE)")
    print("#"*60)
    
    print("\n>>> DISCOUNT LIST (Copy below):")
    print(", ".join([x['ticker'] for x in discount_list]))
    
    print("\n>>> PREMIUM LIST (Copy below):")
    print(", ".join([x['ticker'] for x in premium_list]))

run_scanner()